In [ ]:
import sys
sys.path.append("..")
import utils_for_cluster_first_try as utils
import utils_for_rings_and_discs as gen_and_plot_utils

import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

import importlib
importlib.reload(utils)

from cca_zoo.deep import (
    DCCA,
    DCCA_NOI,
    DCCA_SDL,
    DCCAE,
    DVCCA,    
    architectures
)

In [ ]:
import pickle
import io
importlib.reload(utils)

from pathlib import Path

DATASET_LOAD_PATH = Path.cwd() / "rings_and_discs_dataset.pkl"

with open(DATASET_LOAD_PATH, "rb") as f:
    dataset_dict = pickle.load(f)

### Load the results of the training

In [ ]:
DICT_LOAD_PATH = Path.cwd() / "dict_running_deep_CCA.pkl"

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

with open(DICT_LOAD_PATH, "rb") as f:
    output_dict = CPU_Unpickler(f).load()

In [ ]:
train_dict = output_dict["train_dict"]
losses = train_dict["losses"]

plt.figure(figsize=(10, 6))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss vs Epoch')
plt.grid(True)
plt.show()


model = output_dict["model"]
optimizer = output_dict["optimizer"]

X = dataset_dict["X"]
Y = dataset_dict["Y"]
Z_standard = dataset_dict["Z_standard"]
Z = dataset_dict["Z"]

importlib.reload(utils)

#large validation set
DATASET_LOAD_PATH = Path.cwd() / "rings_and_discs_validation_dataset.pkl"

with open(DATASET_LOAD_PATH, "rb") as f:
    dataset_dict_val = pickle.load(f)


Y_val = dataset_dict_val["Y_val"]
X_val = dataset_dict_val["X_val"]
Z_standard_val = dataset_dict_val["Z_standard_val"]
Z_val = dataset_dict_val["Z_val"]
colors_1_val = Z_standard_val[:,0]
colors_2_val = Z_standard_val[:,1]
val_dataset = utils.XYDataset(X_val,Y_val)

In [ ]:
METHOD_NAME = "DCCA"  # options are "DCCA", "DCCA_NOI", "DCCA_SDL", "DCCAE", "DVCCA"

# with DVCCA there is only one encoder, for Y. So we compute the correlation of U with X, rather than between U and V.
if METHOD_NAME == "DVCCA":
    encoder_Y = model.encoders[0].layers

    U = encoder_Y(torch.tensor(Y).float()).detach().cpu().numpy()
    U_val = encoder_Y(torch.tensor(Y_val).float()).detach().cpu().numpy()
    print("Correlation captured on training set:",utils.compute_correlations(U,X,2))
    
    mycca_out = utils.mycca(X, U, 2)
    if mycca_out is None:
        print("mycca_out is None, no correlation detected on training set.")
        val_correlation = 0
    else:
        print("Correlation captured on training set, according to mycca",mycca_out['S'])
        V_val = X_val @ mycca_out['That']
        U_val_rotated = U_val @ mycca_out['Hhat']
        val_correlation =  [np.corrcoef(U_val_rotated[:, i], V_val[:, i])[0, 1] for i in range(U_val_rotated.shape[1])]
        print("Correlation captured on validation set, according to mycca:",val_correlation)
    
    
else:
    encoder_Y = model.encoders[0]
    encoder_X = model.encoders[1]
    
    U = encoder_Y(torch.tensor(Y).float()).detach().cpu().numpy()
    V = encoder_X(torch.tensor(X).float()).detach().cpu().numpy()
    V_val = encoder_X(torch.tensor(X_val).float()).detach().cpu().numpy()
    U_val = encoder_Y(torch.tensor(Y_val).float()).detach().cpu().numpy()

    cca_out = utils.CCA(V, U, 2)
    if cca_out is None:
        print("cca_out is None, no correlation detected on training set.")
        val_correlation = 0
    else:
        print("Correlation captured on training set:",cca_out['S'])
        V_val_rotated = V_val @ cca_out['T']
        U_val_rotated = U_val @ cca_out['H']
        val_correlation = [np.corrcoef(U_val_rotated[:, i], V_val_rotated[:, i])[0, 1] for i in range(U_val_rotated.shape[1])]
        print("Correlation captured on validation set:",val_correlation)
    
# With these methods, we can also compute the reconstruction error for Y
if METHOD_NAME == "DCCAE" or METHOD_NAME == "DVCCA":
    decoder_Y = model.decoders[0]
    Y_reconstructed = decoder_Y(torch.tensor(U).float()).detach().cpu().numpy()
    Y_val_reconstructed = decoder_Y(torch.tensor(U_val).float()).detach().cpu().numpy()
    recon_error_train = np.mean((Y - Y_reconstructed)**2)
    recon_error_val = np.mean((Y_val - Y_val_reconstructed)**2)
    print("Reconstruction error for Y on training set:", recon_error_train)
    print("Reconstruction error for Y on validation set:", recon_error_val)
    

In [ ]:
print(cca_out['T'])

In [ ]:
importlib.reload(utils)

# we want to plot the latent space
colors_1 = Z_standard[:,2]
colors_2 = Z_standard[:,3]
#utils.plot_latent_variables(U_val_rotated, colors_1, colors_2, color_1_label="radius_of_hole", color_2_label="width_of_disc", H_est=None)

utils.plot_latent_variables_d_greater_than_2(U, Z_standard)

### In-sample correlation attained

In [ ]:
utils.plot_canonical_variables(U - np.mean(U, axis=0), V - np.mean(V, axis=0), colors_1, colors_2)

In [ ]:
importlib.reload(utils)

#out_of_sample_correlations = utils.out_of_sample_correlation(val_dataset,encoder_mean=encoder_mean,H=H_est,T=T_est)
#print("Out-of-sample canonical correlations:", out_of_sample_correlations)
#print("sum:", np.sum(np.abs(out_of_sample_correlations)))


utils.plot_canonical_variables(U_val_rotated - np.mean(U_val_rotated, axis=0), V_val_rotated - np.mean(V_val_rotated, axis=0), colors_1_val, colors_2_val)

#utils.compute_correlations(U_val,V_val,2)
#sum(utils.compute_correlations(U_val,V_val,2))
#print(utils.mycca(X_val,U_val,2))
